# Waste Classification — Organic vs Recyclable

**Transfer learning con VGG16 (Visual Geometry Group 16):** feature extraction → fine-tuning para clasificación binaria de residuos.

Este notebook documenta el proceso exploratorio completo. El código de producción vive en `src/waste_classifier/`.

---

## Contenido

1. Entorno y reproducibilidad
2. Descarga y exploración del dataset
3. Pipeline de datos con `tf.data`
4. Fase 1 — Feature Extraction (extracción de características)
5. Fase 2 — Fine-Tuning (ajuste fino)
6. Evaluación comparativa
7. Grad-CAM (Gradient-weighted Class Activation Mapping)
8. Conclusiones

## 1. Entorno y reproducibilidad

In [ ]:
import os
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# GPU memory growth
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

# Add src/ to path for package imports
sys.path.insert(0, os.path.abspath("../src"))

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices("GPU")}")
print(f"NumPy: {np.__version__}")

## 2. Descarga y exploración del dataset

Dataset: [Waste Classification Data](https://www.kaggle.com/datasets/techsash/waste-classification-data) — 1,200 imágenes con split predefinido train/test, clasificación binaria:
- **O** — Organic (orgánico)
- **R** — Recyclable (reciclable)

In [ ]:
from waste_classifier import download_dataset, PATHS, DATA_CFG

download_dataset()

In [ ]:
# Dataset structure verification
for split in ["train", "test"]:
    for cls in ["O", "R"]:
        p = PATHS.data_dir / split / cls
        n = len(list(p.glob("*")))
        print(f"{split}/{cls}: {n} images")

In [ ]:
# Visual inspection — raw samples before preprocessing
sample_paths = sorted(PATHS.train_dir.glob("O/*"))[:5] + sorted(PATHS.train_dir.glob("R/*"))[:5]

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, path in zip(axes.flat, sample_paths):
    img = tf.keras.utils.load_img(str(path), target_size=DATA_CFG.img_size)
    ax.imshow(img)
    ax.set_title(path.parent.name, fontsize=10)
    ax.axis("off")

axes[0, 0].set_ylabel("Organic", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Recyclable", fontsize=12, fontweight="bold")
plt.suptitle("Raw Samples — No Preprocessing", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Pipeline de datos con 

Se usa  (API moderna, reemplaza al deprecado ) con:

- **Train**: augmentation (flip, translación, rotación, zoom) → rescaling [0,1] → cache → prefetch
- **Val / Test**: rescaling → cache → prefetch

La augmentation se implementa como capas de Keras (), activas solo con .

In [ ]:
from waste_classifier import build_datasets

train_ds, val_ds, test_ds = build_datasets()

print(f"Train batches: {len(train_ds)}")
print(f"Val batches:   {len(val_ds)}")
print(f"Test batches:  {len(test_ds)}")

In [ ]:
# Verify preprocessing
for imgs, labels in train_ds.take(1):
    print(f"Batch shape: {imgs.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Pixel range: [{imgs.numpy().min():.3f}, {imgs.numpy().max():.3f}]")

In [ ]:
# Augmented training samples
label_map = {0: "Organic", 1: "Recyclable"}
sample_imgs, sample_labels = next(iter(train_ds))

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(sample_imgs[i].numpy())
    ax.set_title(label_map[int(sample_labels[i].numpy())], fontsize=10)
    ax.axis("off")
plt.suptitle("Training Samples — After Augmentation & Rescaling", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Fase 1 — Feature Extraction

**Estrategia:** Backbone VGG16 completamente congelado. Solo se entrenan las capas de clasificación (head).

- Optimizer: Adam, LR (Learning Rate) = 1e-4
- Callbacks: EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
- Métrica adicional: AUC (Area Under the Curve)

In [ ]:
from waste_classifier import build_model, print_trainable_summary, TRAIN_CFG

model_ext = build_model(trainable_base=False)
print_trainable_summary(model_ext)
model_ext.summary()

In [ ]:
from waste_classifier import train_phase

hist_ext = train_phase(
    model_ext, train_ds, val_ds,
    learning_rate=TRAIN_CFG.extract_lr,
    checkpoint_name=TRAIN_CFG.extract_checkpoint,
    phase_name="Phase 1 — Feature Extraction",
)

In [ ]:
from waste_classifier import plot_training_curves

plot_training_curves(hist_ext, "Feature Extraction")

**Interpretación:** Las curvas de loss deben mostrar convergencia con la brecha train/val estabilizándose (sin overfitting severo). El accuracy de validación indica la capacidad del head para clasificar con features genéricos de ImageNet.

## 5. Fase 2 — Fine-Tuning

**Estrategia:** Descongelar desde  para adaptar features de alto nivel al dominio de residuos.

- LR reducido (1e-5) para evitar destruir los pesos pre-entrenados
- BatchNormalization se mantiene congelado (sus estadísticas son de ImageNet)

In [ ]:
model_ft = build_model(trainable_base=True, unfreeze_from=TRAIN_CFG.unfreeze_from)
print_trainable_summary(model_ft)

In [ ]:
hist_ft = train_phase(
    model_ft, train_ds, val_ds,
    learning_rate=TRAIN_CFG.finetune_lr,
    checkpoint_name=TRAIN_CFG.finetune_checkpoint,
    phase_name="Phase 2 — Fine-Tuning",
)

In [ ]:
plot_training_curves(hist_ft, "Fine-Tuning")

**Interpretación:** Se espera una mejora marginal sobre feature extraction. Si val_loss diverge, el LR es demasiado alto o se descongelaron demasiadas capas.

## 6. Evaluación comparativa

Métricas sobre el set de test (no visto durante entrenamiento):
- Classification report (precision, recall, F1 por clase)
- Confusion Matrix
- ROC/AUC (Receiver Operating Characteristic / Area Under Curve) comparativo

In [ ]:
from waste_classifier import load_test_images, evaluate_model

test_images, test_labels, test_filenames = load_test_images()
print(f"Test set: {len(test_images)} images ({(test_labels == 0).sum()} Organic, {(test_labels == 1).sum()} Recyclable)")

In [ ]:
res_ext = evaluate_model(model_ext, test_images, test_labels, "Feature Extraction")
res_ft = evaluate_model(model_ft, test_images, test_labels, "Fine-Tuned")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, res, title in [
    (ax1, res_ext, "Feature Extraction"),
    (ax2, res_ft, "Fine-Tuned"),
]:
    cm = confusion_matrix(res["y_true"], res["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Organic", "Recyclable"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{title}\nAUC = {res["auc"]:.3f}", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
from waste_classifier import plot_roc_comparison

plot_roc_comparison({"Feature Extraction": res_ext, "Fine-Tuned": res_ft})

In [ ]:
from waste_classifier import plot_prediction_grid

plot_prediction_grid(model_ft, test_images, test_labels, "Fine-Tuned", n_samples=10)

## 7. Grad-CAM — Explainability

Grad-CAM (Gradient-weighted Class Activation Mapping) visualiza qué regiones de la imagen activaron más la decisión del modelo. Se computa sobre la última capa convolucional ().

In [ ]:
from waste_classifier import plot_grad_cam

for i in [0, 50, 25]:
    plot_grad_cam(model_ft, test_images[i], test_labels[i], "Fine-Tuned")

## 8. Conclusiones

- **Feature extraction** proporciona una baseline sólida con mínimo cómputo.
- **Fine-tuning** mejora la adaptación al dominio específico de residuos al descongelar capas convolutivas de alto nivel.
- **Grad-CAM** confirma que el modelo atiende a las regiones relevantes del objeto, no al fondo.
- VGG16 es adecuado como baseline; para producción real, evaluar EfficientNetV2 o ConvNeXt.

### Próximos pasos

- Aumentar el dataset (el split actual tiene ~1,200 imágenes)
- Experimentar con backbones más eficientes
- Implementar clasificación multi-clase (plástico, vidrio, metal, papel...)
- Optimizar para edge deployment con TFLite (TensorFlow Lite)